In [15]:

# ============================================================
# Attendace Dataset
# ============================================================


import pandas as pd
import re
from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_FILE = Path("data/raw/18-31 Oct 26.xlsx")
OUTPUT_DIR = Path("data/raw_cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "18-31 Oct 26_cleaned.xlsx"

SHEET_NAME = 0

# Dua baris pertama merupakan header utama dan subheader.
HEADER_ROWS = [0, 1]


# ============================================================
# 2. TARGET COLUMN STRUCTURE
#    URUTAN HARUS DIPERTAHANKAN
# ============================================================

TARGET_COLUMNS = [
    "Employee Name",
    "Employee ID",
    "Attendance Date",
    "Day(s)",
    "Shift",
    "Shift In",
    "Shift Out",
    "Roster",
    "Clock-in Time",
    "Clock-out Time",
    "Next Day Clock Time",
    "Clock-in Time (BE)",
    "Clock-out Time (BE)",
    "Attendance (In)",
    "Attendance (Out)",
    "Total Work Hours",
    "Attendance type",
    "Work Group",
    "Enter Overtime (AWH)",
    "Exit Overtime (AWH)",
    "Next Day (OAWH)",
    "Total Overtime Hours (AWF)",
    "Enter Overtime (BWH)",
    "Exit Overtime (BWF)",
    "Next Day (BWH)",
    "Total Overtime Hours (BWH)",
    "Total Overtime Hours",
    "Automatic overtime",
    "Spl No.",
    "SPL Day Type",
    "Difference in work schedule with actual attendance",
    "Difference in check-out time",
    "Status",
    "Check-in attendance terminal",
    "Check-out attendance terminal",
    "Attendance Location",
    "Attendance source",
    "Attendance edit date",
    "Failed attendance posting",
    "Description",
    "Holiday type",
    "Attendance description",
    "Check-in attendance location (mob)",
    "Check-out attendance location (mob)",
    "Check-in attendance address (mob)",
    "Check-out attendance address (mob)",
    "Company Code",
    "Company Name",
    "Position Code",
    "Position Name",
    "Location Code",
    "Location Name",
    "Job Title Code",
    "Job Title Name",
    "Organization Code",
    "Organization Name",
    "Organization Group Code",
    "Organization Group Name",
    "Cost Center Code",
    "Cost Center Name",
    "Job Level Code",
    "Job Level Name",
    "Employment Status",
    "Organization",
    "Job Title",
    "Location",
    "Last Update",
    "Updated By",
]


# ============================================================
# 3. HEADER NORMALIZATION
# ============================================================

def clean_header(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower().startswith("unnamed"):
        return ""

    return re.sub(r"\s+", " ", value)


def normalize(value):

    value = clean_header(value).lower()

    return re.sub(r"[^a-z0-9]", "", value)


# ============================================================
# 4. READ SOURCE DATA
# ============================================================

df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
    header=HEADER_ROWS,
    dtype=object
)

original_row_count = len(df)


# ============================================================
# 5. FLATTEN TWO-LEVEL HEADERS
# ============================================================

source_columns = []

for column in df.columns:

    if isinstance(column, tuple):

        parent = clean_header(column[0])
        child = clean_header(column[1])

    else:

        parent = ""
        child = clean_header(column)

    source_columns.append({
        "parent": parent,
        "child": child,
        "selected": child if child else parent
    })


# ============================================================
# 6. MAP SOURCE HEADERS TO TARGET HEADERS
# ============================================================

def resolve_target(parent, child):

    p = normalize(parent)
    c = normalize(child)

    # --------------------------------------------------------
    # Shift information
    # --------------------------------------------------------

    if c == "shiftin":
        return "Shift In"

    if c == "shiftout":
        return "Shift Out"

    # --------------------------------------------------------
    # Clock-in / Clock-out
    # --------------------------------------------------------

    if c == "clockintime":

        if "beforeediting" in p:
            return "Clock-in Time (BE)"

        return "Clock-in Time"

    if c == "clockouttime":

        if "beforeediting" in p:
            return "Clock-out Time (BE)"

        return "Clock-out Time"

    if c == "nextday":

        if "beforework" in p:
            return "Next Day (BWH)"

        if "afterwork" in p:
            return "Next Day (OAWH)"

        return "Next Day Clock Time"

    # --------------------------------------------------------
    # Overtime mapping
    # --------------------------------------------------------

    overtime_group = None

    if "afterwork" in p:
        overtime_group = "AWH"

    elif "beforework" in p:
        overtime_group = "BWH"

    if c == "enterovertime" and overtime_group:
        return f"Enter Overtime ({overtime_group})"

    if c == "exitovertime" and overtime_group:
        return f"Exit Overtime ({overtime_group})"

    if c == "totalovertimehours":

        if "afterwork" in p:
            return "Total Overtime Hours (AWF)"

        if "beforework" in p:
            return "Total Overtime Hours (BWH)"

        return "Total Overtime Hours"

    # --------------------------------------------------------
    # Holiday mapping
    # --------------------------------------------------------

    if c == "holiday":
        return "Holiday type"

    # --------------------------------------------------------
    # Mobile attendance location mapping
    # --------------------------------------------------------

    if c == "checkinattendancelocation":
        return "Check-in attendance location (mob)"

    if c == "checkoutattendancelocation":
        return "Check-out attendance location (mob)"

    if c == "checkinattendanceaddress":
        return "Check-in attendance address (mob)"

    if c == "checkoutattendanceaddress":
        return "Check-out attendance address (mob)"

    # --------------------------------------------------------
    # Difference mapping
    # --------------------------------------------------------

    if c == "differenceincheckouttime":
        return "Difference in check-out time"

    if c == "differenceinworkschedulewithactualattendance":
        return "Difference in work schedule with actual attendance"

    # --------------------------------------------------------
    # General header mapping
    # --------------------------------------------------------

    aliases = {
        "employeename": "Employee Name",
        "employeeid": "Employee ID",
        "attendancedate": "Attendance Date",
        "days": "Day(s)",
        "shift": "Shift",
        "roster": "Roster",
        "attendancein": "Attendance (In)",
        "attendanceout": "Attendance (Out)",
        "totalworkhours": "Total Work Hours",
        "attendancetype": "Attendance type",
        "workgroup": "Work Group",
        "automaticovertime": "Automatic overtime",
        "splno": "Spl No.",
        "spldaytype": "SPL Day Type",
        "status": "Status",
        "checkinattendanceterminal": "Check-in attendance terminal",
        "checkoutattendanceterminal": "Check-out attendance terminal",
        "attendancelocation": "Attendance Location",
        "attendancesource": "Attendance source",
        "attendanceeditdate": "Attendance edit date",
        "failedattendanceposting": "Failed attendance posting",
        "description": "Description",
        "holidaytype": "Holiday type",
        "attendancedescription": "Attendance description",
        "companycode": "Company Code",
        "companyname": "Company Name",
        "positioncode": "Position Code",
        "positionname": "Position Name",
        "locationcode": "Location Code",
        "locationname": "Location Name",
        "jobtitlecode": "Job Title Code",
        "jobtitlename": "Job Title Name",
        "organizationcode": "Organization Code",
        "organizationname": "Organization Name",
        "organizationgroupcode": "Organization Group Code",
        "organizationgroupname": "Organization Group Name",
        "costcentercode": "Cost Center Code",
        "costcentername": "Cost Center Name",
        "joblevelcode": "Job Level Code",
        "joblevelname": "Job Level Name",
        "employmentstatus": "Employment Status",
        "organization": "Organization",
        "jobtitle": "Job Title",
        "location": "Location",
        "lastupdate": "Last Update",
        "updatedby": "Updated By",
    }

    return aliases.get(c, aliases.get(p))


# ============================================================
# 7. BUILD TARGET DATAFRAME
# ============================================================

result = pd.DataFrame(index=df.index)

# Semua target dibuat terlebih dahulu.
# Kolom yang tidak ditemukan akan tetap kosong.

for column in TARGET_COLUMNS:
    result[column] = pd.NA


# Menyimpan informasi mapping untuk pemeriksaan.
mapping_report = []
unmapped_columns = []

for position, source in enumerate(source_columns):

    parent = source["parent"]
    child = source["child"]

    target = resolve_target(parent, child)

    source_name = (
        f"{parent} / {child}"
        if parent and child
        else child or parent
    )

    if target in TARGET_COLUMNS:

        if result[target].notna().any():
            # Hindari menimpa data yang sudah berhasil dipetakan.
            unmapped_columns.append(source_name)
            continue

        result[target] = df.iloc[:, position]

        mapping_report.append({
            "Source Column": source_name,
            "Target Column": target,
            "Source Position": position
        })

    elif source_name:

        unmapped_columns.append(source_name)


# ============================================================
# 8. DATA INTEGRITY CHECK
# ============================================================

assert len(result) == original_row_count, (
    "Jumlah baris berubah!"
)

assert list(result.columns) == TARGET_COLUMNS, (
    "Struktur kolom tidak sesuai!"
)

# Pemeriksaan kolom kosong.
empty_columns = [
    column
    for column in TARGET_COLUMNS
    if result[column].isna().all()
]


# ============================================================
# 9. EXPORT
# ============================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    result.to_excel(
        writer,
        sheet_name="Attendance",
        index=False
    )

    pd.DataFrame(mapping_report).to_excel(
        writer,
        sheet_name="Mapping Report",
        index=False
    )

    pd.DataFrame({
        "Empty Target Columns": empty_columns
    }).to_excel(
        writer,
        sheet_name="Empty Columns",
        index=False
    )

    pd.DataFrame({
        "Unmapped Source Columns": unmapped_columns
    }).to_excel(
        writer,
        sheet_name="Unmapped Columns",
        index=False
    )


# ============================================================
# 10. FINAL REPORT
# ============================================================

print("=" * 55)
print("ATTENDANCE RESTRUCTURING COMPLETED")
print("=" * 55)

print(f"Original rows : {original_row_count:,}")
print(f"Output rows   : {len(result):,}")

print(f"Source columns: {len(df.columns)}")
print(f"Target columns: {len(result.columns)}")

print(f"\nOutput file: {OUTPUT_FILE}")

print(f"\nEmpty target columns: {len(empty_columns)}")
print(f"Unmapped source columns: {len(unmapped_columns)}")

print("\nAll target columns:")
for index, column in enumerate(result.columns, start=1):
    print(f"{index:02d}. {column}")

print("\nRestructuring finished.")


ATTENDANCE RESTRUCTURING COMPLETED
Original rows : 58,963
Output rows   : 58,963
Source columns: 68
Target columns: 68

Output file: data\raw_cleaned\18-31 Oct 26_cleaned.xlsx

Empty target columns: 19
Unmapped source columns: 3

All target columns:
01. Employee Name
02. Employee ID
03. Attendance Date
04. Day(s)
05. Shift
06. Shift In
07. Shift Out
08. Roster
09. Clock-in Time
10. Clock-out Time
11. Next Day Clock Time
12. Clock-in Time (BE)
13. Clock-out Time (BE)
14. Attendance (In)
15. Attendance (Out)
16. Total Work Hours
17. Attendance type
18. Work Group
19. Enter Overtime (AWH)
20. Exit Overtime (AWH)
21. Next Day (OAWH)
22. Total Overtime Hours (AWF)
23. Enter Overtime (BWH)
24. Exit Overtime (BWF)
25. Next Day (BWH)
26. Total Overtime Hours (BWH)
27. Total Overtime Hours
28. Automatic overtime
29. Spl No.
30. SPL Day Type
31. Difference in work schedule with actual attendance
32. Difference in check-out time
33. Status
34. Check-in attendance terminal
35. Check-out attendance

In [14]:

# ============================================================
# Display Dataset
# ============================================================


import pandas as pd
from pathlib import Path
from openpyxl import load_workbook

# ==========================================
# 1. CONFIGURATION
# ==========================================

TEMPLATE_FILE = Path("data/raw/Cuti.xlsx")

SOURCE_FILE = Path(
    "data/raw/Cuti (01 Oc - 09 Nov).xlsx"
)

OUTPUT_DIR = Path("data/raw/Cuti (01 Oc - 09 Nov)_standardized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "Cuti_standardized.xlsx"

TEMPLATE_SHEET = "Attendance Display (Cal)"
SOURCE_SHEET = "Attendance Display"

# File sumber memiliki 2 baris header
SOURCE_HEADER_ROWS = 2

# Kolom sumber yang tidak memiliki pasangan
# yang sesuai pada struktur Cuti.xlsx
EXTRA_SOURCE_COLUMNS = {
    16: "Late",
    17: "Early",
    18: "Check-in attendance terminal",
    19: "Check-out attendance terminal",
    20: "Attendance Location",
}


# ==========================================
# 2. READ TEMPLATE HEADER
# ==========================================

workbook = load_workbook(
    TEMPLATE_FILE,
    read_only=True,
    data_only=True
)

worksheet = workbook[TEMPLATE_SHEET]

TARGET_COLUMNS = [
    cell.value if cell.value is not None else ""
    for cell in worksheet[1]
]

workbook.close()

if len(TARGET_COLUMNS) != 83:
    raise ValueError(
        f"Template memiliki {len(TARGET_COLUMNS)} kolom, "
        "seharusnya 83 kolom."
    )


# ==========================================
# 3. READ SOURCE FILE
# ==========================================

source_raw = pd.read_excel(
    SOURCE_FILE,
    sheet_name=SOURCE_SHEET,
    header=None,
    dtype=object
)

if source_raw.shape[1] != len(TARGET_COLUMNS):
    raise ValueError(
        f"Jumlah kolom sumber ({source_raw.shape[1]}) "
        f"tidak sama dengan template ({len(TARGET_COLUMNS)})."
    )

# Lewati dua baris header
source_data = source_raw.iloc[
    SOURCE_HEADER_ROWS:
].copy()

source_data.reset_index(drop=True, inplace=True)

original_row_count = len(source_data)


# ==========================================
# 4. SAVE SOURCE-ONLY COLUMNS
# ==========================================

extra_data = pd.DataFrame({
    "Source Row Number": range(
        SOURCE_HEADER_ROWS + 1,
        SOURCE_HEADER_ROWS + 1 + original_row_count
    ),
    "Employee ID": source_data.iloc[:, 1].values,
    "Attendance Date": source_data.iloc[:, 2].values,
})

for col_idx, col_name in EXTRA_SOURCE_COLUMNS.items():
    extra_data[col_name] = source_data.iloc[:, col_idx].values


# ==========================================
# 5. STANDARDIZE STRUCTURE
# ==========================================

standardized = source_data.copy()

# Kosongkan kolom yang tidak sesuai dengan
# struktur template Cuti.xlsx.
for col_idx in EXTRA_SOURCE_COLUMNS:
    standardized.iloc[:, col_idx] = pd.NA

# Gunakan nama dan urutan kolom template.
# Posisi kolom tetap dipertahankan, termasuk
# nama kolom duplikat dan header kosong.
standardized.columns = TARGET_COLUMNS


# ==========================================
# 6. VALIDATION
# ==========================================

print("\n========== VALIDATION ==========")

print("Original rows :", original_row_count)
print("Output rows   :", len(standardized))
print("Target columns:", len(TARGET_COLUMNS))
print("Output columns:", len(standardized.columns))

if len(standardized) != original_row_count:
    raise ValueError(
        "ERROR: Jumlah baris berubah!"
    )

if len(standardized.columns) != len(TARGET_COLUMNS):
    raise ValueError(
        "ERROR: Struktur kolom tidak sesuai template!"
    )


# ==========================================
# 7. EXPORT
# ==========================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    standardized.to_excel(
        writer,
        sheet_name="Attendance",
        index=False
    )

    extra_data.to_excel(
        writer,
        sheet_name="Source Extra Columns",
        index=False
    )


print("\nStandardization completed.")
print("Output:", OUTPUT_FILE)
print("Rows preserved:", len(standardized))


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_784\1516460289.py:90: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  source_data.reset_index(drop=True, inplace=True)



========== VALIDATION ==========
Original rows : 30295
Output rows   : 30295
Target columns: 83
Output columns: 83

Standardization completed.
Output: data\raw\Cuti (01 Oc - 09 Nov)_standardized\Cuti_standardized.xlsx
Rows preserved: 30295


In [ ]:

# ============================================================
# Added Dataset
# ============================================================


import pandas as pd
from pathlib import Path

# =================================
# 1. CONFIGURATION
# =================================

INPUT_DIR = Path("data/raw_cleaned")
OUTPUT_DIR = Path("data/raw_cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "combined.xlsx"

FILES = [
    INPUT_DIR / "1-11 Nov 26_cleaned.xlsx",
    INPUT_DIR / "1-17 Oct 26_cleaned.xlsx",
    INPUT_DIR / "18-31 Oct 26_cleaned.xlsx"
]

# =================================
# 2. READ AND COMBINE FILES
# =================================

dataframes = []

for file in FILES:
    df = pd.read_excel(file)

    # Tambahkan identitas sumber file
    df["source_file"] = file.name

    dataframes.append(df)

combined = pd.concat(
    dataframes,
    ignore_index=True
)

# =================================
# 3. EXPORT
# =================================

combined.to_excel(OUTPUT_FILE, index=False)

print(f"Total rows: {len(combined):,}")
print(f"Total columns: {len(combined.columns)}")
print(f"Output saved: {OUTPUT_FILE}")


Total rows: 171,605
Total columns: 69
Output saved: raw_cleaned\combined.xlsx
